In [2]:
import spot

class SpotLTLMutator:
    # 1. Map Spot's built-in operator kinds to your category sets
    # spot.op_G (□), spot.op_F (♢), spot.op_X (◯), spot.op_Not (¬)
    UNARY_KINDS = {spot.op_G, spot.op_F, spot.op_X, spot.op_Not}
    
    # spot.op_Or (∨), spot.op_And (∧), spot.op_U (U), spot.op_R (R), spot.op_W (W)
    BINARY_KINDS = {spot.op_Or, spot.op_And, spot.op_U, spot.op_R, spot.op_W}
    
    # Allowed binary operators for Rule 3(d)
    RULE_3D_OPS = {spot.op_U, spot.op_W, spot.op_And, spot.op_Or}

    def __init__(self, atomic_propositions: list):
        """Initialize with a list of AP strings, e.g., ['p', 'q']"""
        self.ap_strings = set(atomic_propositions)
        self.ap_formulas = [spot.formula.ap(p) for p in atomic_propositions]
        self.constants = [spot.formula_ff(), spot.formula_tt()] # false, true

    def mutate(self, phi: spot.formula) -> list:
        """
        Main entry point. Takes a spot.formula object and returns 
        a deduplicated list of mutated spot.formula objects.
        """
        mutations = []
        phi_kind = phi.kind()

        # --- Structural Check Using Pure Explicit Kinds ---
        if phi_kind == spot.op_tt or phi_kind == spot.op_ff:
            mutations.extend(self._get_base_cases(phi, is_constant=True))
            
        elif phi_kind == spot.op_ap:
            mutations.extend(self._get_base_cases(phi, is_constant=False))
            
        elif phi_kind in self.UNARY_KINDS: 
            mutations.extend(self._get_unary_cases(phi))
            
        elif phi_kind in self.BINARY_KINDS: 
            mutations.extend(self._get_binary_cases(phi))

        # --- Apply General Cases (Rules 5 & 6) ---
        mutations.extend(self._get_general_cases(phi))

        # Deduplicate safely using unique string representations
        seen = set()
        deduped = []
        for m in mutations:
            f_str = m.to_str()
            if f_str not in seen:
                seen.add(f_str)
                deduped.append(m)
                
        return deduped

    def _get_base_cases(self, phi, is_constant: bool) -> list:
        """Handles Rules 1 and 2"""
        mutations = []
        if is_constant:
            # Rule 1: true -> false; false -> true
            mutations.append(spot.formula_ff() if phi.kind() == spot.op_tt else spot.formula_tt())
        else:
            # Rule 2: If p -> q where p != q
            p_str = phi.to_str()
            for q in self.ap_formulas:
                if q.to_str() != p_str:
                    mutations.append(q)
        return mutations

    def _get_unary_cases(self, phi: spot.formula) -> list:
        """Handles Rule 3 (Inductive Unary)"""
        mutations = []
        op1 = phi.kind()
        phi1 = phi[0]  # Safely get structural sub-operand
        
        # (a) phi' = op1' phi1
        for op1_prime in self.UNARY_KINDS:
            if op1 != op1_prime:
                mutations.append(spot.formula_unop(op1_prime, phi1))
                
        # (b) phi' = phi1
        mutations.append(phi1)
        
        # (c) phi' = op1 mutate(phi1)
        for m in self.mutate(phi1):
            mutations.append(spot.formula_unop(op1, m))
            
        # (d) phi' = p op2' phi
        for p in self.ap_formulas:
            for op2_prime in self.RULE_3D_OPS:
                mutations.append(spot.formula_binop(op2_prime, p, phi))
                
        return mutations

    def _get_binary_cases(self, phi: spot.formula) -> list:
        """Handles Rule 4 (Inductive Binary)"""
        mutations = []
        op2 = phi.kind()
        phi1 = phi[0]  # Left sub-operand
        phi2 = phi[1]  # Right sub-operand
        
        # (a) phi' = phi1 op2' phi2
        for op2_prime in self.BINARY_KINDS:
            if op2 != op2_prime:
                mutations.append(spot.formula_binop(op2_prime, phi1, phi2))
                
        # (b) phi' = phi_i
        mutations.append(phi1)
        mutations.append(phi2)
        
        # (c) phi' = mutate(phi1) op2 phi2
        for m in self.mutate(phi1):
            mutations.append(spot.formula_binop(op2, m, phi2))
            
        # (d) phi' = phi1 op2 mutate(phi2)
        for m in self.mutate(phi2):
            mutations.append(spot.formula_binop(op2, phi1, m))
            
        return mutations

    def _get_general_cases(self, phi: spot.formula) -> list:
        """Handles Rules 5 and 6 (General Cases)"""
        mutations = []
        
        # Rule 5: wrap the whole formula in a unary operator
        for op1 in self.UNARY_KINDS:
            mutations.append(spot.formula_unop(op1, phi))
            
        # Rule 6: Replace with true, false, or any AP
        mutations.extend(self.constants)
        mutations.extend(self.ap_formulas)
        
        return mutations


# --- Example Tester Verification ---
if __name__ == "__main__":
    mutator = SpotLTLMutator(atomic_propositions=["p", "q"])
    
    # Parse a test formula string cleanly using Spot syntax
    input_formula = spot.formula("G(p | F q)")
    
    print(f"Original Spot Formula: {input_formula}")
    print("-" * 50)
    
    results = mutator.mutate(input_formula)
    print(f"Generated {len(results)} distinct mutants via Spot API:\n")
    
    for m in sorted(results, key=lambda x: x.to_str())[:15]: 
        print(f"  -> {m.to_str()}")

: 

In [1]:
{spot.op_G, spot.op_F, spot.op_X, spot.op_Not}
{spot.op_Or, spot.op_And, spot.op_U, spot.op_R, spot.op_W}
{spot.op_U, spot.op_W, spot.op_And, spot.op_Or}

NameError: name 'spot' is not defined

In [6]:
import spot


class LTLMutator:
    """
    Definition 9 LTL mutator.

    Given a Spot formula φ, returns ALL possible one-step mutations φ'
    allowed by the definition.
    """

    UNARY_OPS = ["!", "X", "F", "G"]
    BINARY_OPS = ["|", "&", "U", "R", "W"]

    def __init__(self, ap_set=None):
        """
        Parameters
        ----------
        ap_set : iterable[str] or None

            AP from the definition.

            If None, AP is inferred from the formula.
        """
        self.ap_set = set(ap_set) if ap_set is not None else None

    # ============================================================
    # Public API
    # ============================================================

    def mutate(self, formula):
        """
        Return all possible mutations of formula.
        """
        aps = self.ap_set or self.extract_aps(formula)

        mutations = set()

        # General cases (5) and (6)
        mutations |= self.general_case_5(formula)
        mutations |= self.general_case_6(aps)

        # Syntax-directed cases
        mutations |= self._mutate_recursive(formula, aps)

        mutations.discard(str(formula))
        return {spot.formula(m) for m in mutations}

    # ============================================================
    # AP extraction
    # ============================================================

    def extract_aps(self, formula):
        aps = set()

        def visit(f):
            if f.kind() == spot.op_ap:
                aps.add(f.ap_name())
            else:
                for c in f:
                    visit(c)

        visit(formula)
        return aps

    # ============================================================
    # Recursive mutation dispatcher
    # ============================================================

    def _mutate_recursive(self, phi, aps):
        k = phi.kind()

        if k == spot.op_tt:
            return self.base_case_true()

        if k == spot.op_ff:
            return self.base_case_false()

        if k == spot.op_ap:
            return self.base_case_ap(phi, aps)

        unary_map = {
            spot.op_Not: "!",
            spot.op_X: "X",
            spot.op_F: "F",
            spot.op_G: "G",
        }

        if k in unary_map:
            return self.unary_cases(phi, unary_map[k], aps)

        binary_map = {
            spot.op_Or: "|",
            spot.op_And: "&",
            spot.op_U: "U",
            spot.op_R: "R",
            spot.op_W: "W",
        }

        if k in binary_map:
            return self.binary_cases(phi, binary_map[k], aps)

        return set()

    # ============================================================
    # Base cases
    # ============================================================

    def base_case_true(self):
        # Definition 9.1
        return {"false"}

    def base_case_false(self):
        # Definition 9.1
        return {"true"}

    def base_case_ap(self, phi, aps):
        # Definition 9.2
        p = phi.ap_name()

        return {
            q
            for q in aps
            if q != p
        }

    # ============================================================
    # Case 3: unary formulas
    # ============================================================

    def unary_cases(self, phi, op, aps):
        child = phi[0]

        results = set()

        # 3(a)
        results |= self.unary_replace_operator(op, child)

        # 3(b)
        results.add(str(child))

        # 3(c)
        results |= self.unary_mutate_child(op, child, aps)

        # 3(d)
        results |= self.unary_extend_with_binary(phi, aps)

        return results

    def unary_replace_operator(self, op, child):
        result = set()

        for op2 in self.UNARY_OPS:
            if op2 != op:
                result.add(f"{op2}({child})")

        return result

    def unary_mutate_child(self, op, child, aps):
        result = set()

        for m in self._mutate_recursive(child, aps):
            result.add(f"{op}({m})")

        return result

    def unary_extend_with_binary(self, phi, aps):
        result = set()

        for p in aps:
            for bop in ["U", "W", "&", "|"]:
                result.add(f"({p}) {bop} ({phi})")

        return result

    # ============================================================
    # Case 4: binary formulas
    # ============================================================

    def binary_cases(self, phi, op, aps):
        left = phi[0]
        right = phi[1]

        results = set()

        # 4(a)
        results |= self.binary_replace_operator(left, op, right)

        # 4(b)
        results.add(str(left))
        results.add(str(right))

        # 4(c)
        results |= self.binary_mutate_left(left, op, right, aps)

        # 4(d)
        results |= self.binary_mutate_right(left, op, right, aps)

        return results

    def binary_replace_operator(self, left, op, right):
        result = set()

        for op2 in self.BINARY_OPS:
            if op2 != op:
                result.add(f"({left}) {op2} ({right})")

        return result

    def binary_mutate_left(self, left, op, right, aps):
        result = set()

        for m in self._mutate_recursive(left, aps):
            result.add(f"({m}) {op} ({right})")

        return result

    def binary_mutate_right(self, left, op, right, aps):
        result = set()

        for m in self._mutate_recursive(right, aps):
            result.add(f"({left}) {op} ({m})")

        return result

    # ============================================================
    # General cases
    # ============================================================

    def general_case_5(self, phi):
        result = set()

        for op in self.UNARY_OPS:
            result.add(f"{op}({phi})")

        return result

    def general_case_6(self, aps):
        result = {"true", "false"}

        result.update(aps)

        return result

In [4]:
f = spot.formula("G(a U b)")

mutator = LTLMutator()

mutations = mutator.mutate(f)

print(f"Number of mutations: {len(mutations)}")
for m in sorted(map(str, mutations)):
    print(m)

Number of mutations: 26
!(a U b)
!G(a U b)
0
1
F(a U b)
FG(a U b)
G(a & b)
G(a R b)
G(a U b)
G(a W b)
G(a | b)
Ga
Gb
X(a U b)
XG(a U b)
a
a & G(a U b)
a U G(a U b)
a U b
a W G(a U b)
a | G(a U b)
b
b & G(a U b)
b U G(a U b)
b W G(a U b)
b | G(a U b)


In [7]:
from collections import deque


def mutation_distance_bidirectional(
    source,
    target,
    mutator,
    max_depth=20
):
    s = str(source)
    t = str(target)

    if s == t:
        return 0

    front_a = {s: 0}
    front_b = {t: 0}

    queue_a = deque([source])
    queue_b = deque([target])

    while queue_a and queue_b:

        if len(queue_a) <= len(queue_b):

            for _ in range(len(queue_a)):
                current = queue_a.popleft()

                d = front_a[str(current)]

                if d >= max_depth:
                    continue

                for nxt in mutator.mutate(current):

                    ns = str(nxt)

                    if ns in front_b:
                        return d + 1 + front_b[ns]

                    if ns not in front_a:
                        front_a[ns] = d + 1
                        queue_a.append(nxt)

        else:

            for _ in range(len(queue_b)):
                current = queue_b.popleft()

                d = front_b[str(current)]

                if d >= max_depth:
                    continue

                for nxt in mutator.mutate(current):

                    ns = str(nxt)

                    if ns in front_a:
                        return d + 1 + front_a[ns]

                    if ns not in front_b:
                        front_b[ns] = d + 1
                        queue_b.append(nxt)

    return None

In [8]:
import spot
f1 = spot.formula("G(a & (F b)) -> ((!c) U d)")
f2 = spot.formula("F(a)& (G b) -> ((!d) U b)")

mutator = LTLMutator()

d = mutation_distance_bidirectional(f1, f2, mutator)

print(d)

2


In [12]:

mutator = LTLMutator()

mutations = mutator.mutate(f2)

print(f"Number of mutations: {len(mutations)}")
for m in sorted(map(str, mutations)):
    print(m)


Number of mutations: 9
!((Fa & Gb) -> (!d U b))
0
1
F((Fa & Gb) -> (!d U b))
G((Fa & Gb) -> (!d U b))
X((Fa & Gb) -> (!d U b))
a
b
d


In [1]:
import spot

def mutate_ltl(phi: spot.formula, AP: set) -> set:
    """
    Mutates an LTL formula based on Definition 9.
    Returns a set of mutated spot.formula objects.
    """
    mutants = set()
    
    # ---------------------------------------------------------
    # Helper definitions for Spot operator mappings
    # ---------------------------------------------------------
    unary_factories = {
        spot.op_Not: spot.formula.Not,
        spot.op_X:   spot.formula.X,    # Next / Circle
        spot.op_F:   spot.formula.F,    # Eventually / Diamond
        spot.op_G:   spot.formula.G     # Always / Square
    }
    
    # Note: Spot's And/Or are n-ary. We use lists to construct them.
    binary_factories = {
        spot.op_U:   spot.formula.U,
        spot.op_R:   spot.formula.R,
        spot.op_W:   spot.formula.W,
        spot.op_And: lambda left, right: spot.formula.And([left, right]),
        spot.op_Or:  lambda left, right: spot.formula.Or([left, right])
    }

    # ==========================================
    # GENERAL CASES (Rules 5 & 6)
    # ==========================================
    
    # Rule 5: Wrap in Unary
    for factory in unary_factories.values():
        mutants.add(factory(phi))
        
    # Rule 6: Replace with constants or APs
    mutants.add(spot.formula.tt()) # true
    mutants.add(spot.formula.ff()) # false
    for p in AP:
        mutants.add(spot.formula.ap(p))

    # ==========================================
    # BASE & INDUCTIVE CASES
    # ==========================================
    kind = phi.kind()

    # Base Rule 1: Constants
    if kind == spot.op_tt:
        mutants.add(spot.formula.ff())
    elif kind == spot.op_ff:
        mutants.add(spot.formula.tt())
        
    # Base Rule 2: Atomic Propositions
    elif kind == spot.op_ap:
        p_name = phi.ap_name()
        for q in AP:
            if q != p_name:
                mutants.add(spot.formula.ap(q))

    # Inductive Rule 3: Unary Operators
    elif kind in unary_factories:
        child = phi[0] # Get the single child of the unary operator
        
        # (a) Swap operator
        for op, factory in unary_factories.items():
            if op != kind:
                mutants.add(factory(child))
                
        # (b) Drop operator entirely
        mutants.add(child)
        
        # (c) Mutate child
        for m in mutate_ltl(child, AP):
            mutants.add(unary_factories[kind](m))
            
        # (d) Prepend Binary: p op_2' phi
        for p in AP:
            for op, factory in binary_factories.items():
                if op in [spot.op_U, spot.op_W, spot.op_And, spot.op_Or]:
                    mutants.add(factory(spot.formula.ap(p), phi))

    # Inductive Rule 4: Binary Operators
    # Warning: Spot collapses (a & b & c) into a single n-ary node. 
    # For strict adherence, we treat the first child as 'left' and the rest as 'right'.
    elif kind in binary_factories or kind in [spot.op_And, spot.op_Or]:
        if phi.size() >= 2:
            left = phi[0]
            # Handle n-ary Or/And by grouping the remaining children
            if kind == spot.op_And and phi.size() > 2:
                right = spot.formula.And([phi[i] for i in range(1, phi.size())])
            elif kind == spot.op_Or and phi.size() > 2:
                right = spot.formula.Or([phi[i] for i in range(1, phi.size())])
            else:
                right = phi[1]
            
            # (a) Swap operator
            for op, factory in binary_factories.items():
                if op != kind:
                    mutants.add(factory(left, right))
                    
            # (b) Drop operator (yield children)
            mutants.add(left)
            mutants.add(right)
            
            # (c) Mutate left child
            for m in mutate_ltl(left, AP):
                mutants.add(binary_factories.get(kind, binary_factories[spot.op_And])(m, right))
                
            # (d) Mutate right child
            for m in mutate_ltl(right, AP):
                mutants.add(binary_factories.get(kind, binary_factories[spot.op_And])(left, m))

    return mutants

In [6]:
# --- Usage Example ---

formula_str = "G(p -> F q)"

# 1. Parse the formula
phi = spot.formula(formula_str)
print(f"Original Formula: {phi}\n")

# 2. Extract Atomic Propositions automatically
# spot.atomic_prop_collect returns a set of spot.formula objects, 
# we just want their string names for our AP set.
ap_set = {ap.ap_name() for ap in spot.atomic_prop_collect(phi)}

# If you want to introduce new APs not in the formula, add them here
# ap_set.add("r")

# 3. Generate Mutants
mutants = mutate_ltl(phi, ap_set)

# 4. Print Results
print(f"Generated {len(mutants)} mutants:")
for idx, m in enumerate(mutants, 1):
    # Spot automatically formats standard LTL strings
    print(f"{idx:3d}: {m}")
    
# Optional: Filter out semantically equivalent mutants
print("\n--- Applying Simplification ---")
simplified_mutants = {spot.simplify(m) for m in mutants}
print(f"Reduced to {len(simplified_mutants)} unique behavioral formulas.")

Original Formula: G(p -> Fq)

Generated 25 mutants:
  1: 0
  2: 1
  3: p
  4: q
  5: p -> Fq
  6: G(p -> Fq)
  7: !G(p -> Fq)
  8: XG(p -> Fq)
  9: FG(p -> Fq)
 10: !(p -> Fq)
 11: X(p -> Fq)
 12: F(p -> Fq)
 13: Gp
 14: Gq
 15: G!(p -> Fq)
 16: GX(p -> Fq)
 17: GF(p -> Fq)
 18: p U G(p -> Fq)
 19: p W G(p -> Fq)
 20: p & G(p -> Fq)
 21: p | G(p -> Fq)
 22: q U G(p -> Fq)
 23: q W G(p -> Fq)
 24: q & G(p -> Fq)
 25: q | G(p -> Fq)

--- Applying Simplification ---
Reduced to 24 unique behavioral formulas.


In [10]:
simplified_mutants

{spot.formula("0"),
 spot.formula("1"),
 spot.formula("p"),
 spot.formula("q"),
 spot.formula("Gp"),
 spot.formula("Gq"),
 spot.formula("!p | Fq"),
 spot.formula("G(!p | Fq)"),
 spot.formula("p & G!q"),
 spot.formula("F(p & G!q)"),
 spot.formula("XG(!p | Fq)"),
 spot.formula("FG(!p | Fq)"),
 spot.formula("X(!p | Fq)"),
 spot.formula("F(!p | q)"),
 spot.formula("G(p & !q)"),
 spot.formula("GF(!p | q)"),
 spot.formula("p U G(!p | Fq)"),
 spot.formula("p W G(!p | Fq)"),
 spot.formula("p & G(!p | Fq)"),
 spot.formula("p | G(!p | Fq)"),
 spot.formula("q U G(!p | Fq)"),
 spot.formula("q W G(!p | Fq)"),
 spot.formula("q & G(!p | Fq)"),
 spot.formula("q | G(!p | Fq)")}

In [14]:
formula_str = "G(p -> F q)"
formula_str = "G(!p | F q)"
phi = spot.formula(formula_str) 
AP =  set({'p', 'q'})

mutants = set()
    
# ---------------------------------------------------------
# Helper definitions for Spot operator mappings
# ---------------------------------------------------------
unary_factories = {
    spot.op_Not: spot.formula.Not,
    spot.op_X:   spot.formula.X,    # Next / Circle
    spot.op_F:   spot.formula.F,    # Eventually / Diamond
    spot.op_G:   spot.formula.G     # Always / Square
}

# Note: Spot's And/Or are n-ary. We use lists to construct them.
binary_factories = {
    spot.op_U:   spot.formula.U,
    spot.op_R:   spot.formula.R,
    spot.op_W:   spot.formula.W,
    spot.op_And: lambda left, right: spot.formula.And([left, right]),
    spot.op_Or:  lambda left, right: spot.formula.Or([left, right])
}

# ==========================================
# GENERAL CASES (Rules 5 & 6)
# ==========================================

# Rule 5: Wrap in Unary
for factory in unary_factories.values():
    mutants.add(factory(phi))
    
# Rule 6: Replace with constants or APs
mutants.add(spot.formula.tt()) # true
mutants.add(spot.formula.ff()) # false
for p in AP:
    mutants.add(spot.formula.ap(p))

# ==========================================
# BASE & INDUCTIVE CASES
# ==========================================
kind = phi.kind()

# Base Rule 1: Constants
if kind == spot.op_tt:
    mutants.add(spot.formula.ff())
elif kind == spot.op_ff:
    mutants.add(spot.formula.tt())
    
# Base Rule 2: Atomic Propositions
elif kind == spot.op_ap:
    p_name = phi.ap_name()
    for q in AP:
        if q != p_name:
            mutants.add(spot.formula.ap(q))

# Inductive Rule 3: Unary Operators
elif kind in unary_factories:
    child = phi[0] # Get the single child of the unary operator
    
    # (a) Swap operator
    for op, factory in unary_factories.items():
        if op != kind:
            mutants.add(factory(child))
            
    # (b) Drop operator entirely
    mutants.add(child)
    
    # (c) Mutate child
    for m in mutate_ltl(child, AP):
        mutants.add(unary_factories[kind](m))
        
    # (d) Prepend Binary: p op_2' phi
    for p in AP:
        for op, factory in binary_factories.items():
            if op in [spot.op_U, spot.op_W, spot.op_And, spot.op_Or]:
                mutants.add(factory(spot.formula.ap(p), phi))

# Inductive Rule 4: Binary Operators
# Warning: Spot collapses (a & b & c) into a single n-ary node. 
# For strict adherence, we treat the first child as 'left' and the rest as 'right'.
elif kind in binary_factories or kind in [spot.op_And, spot.op_Or]:
    if phi.size() >= 2:
        left = phi[0]
        # Handle n-ary Or/And by grouping the remaining children
        if kind == spot.op_And and phi.size() > 2:
            right = spot.formula.And([phi[i] for i in range(1, phi.size())])
        elif kind == spot.op_Or and phi.size() > 2:
            right = spot.formula.Or([phi[i] for i in range(1, phi.size())])
        else:
            right = phi[1]
        
        # (a) Swap operator
        for op, factory in binary_factories.items():
            if op != kind:
                mutants.add(factory(left, right))
                
        # (b) Drop operator (yield children)
        mutants.add(left)
        mutants.add(right)
        
        # (c) Mutate left child
        for m in mutate_ltl(left, AP):
            mutants.add(binary_factories.get(kind, binary_factories[spot.op_And])(m, right))
            
        # (d) Mutate right child
        for m in mutate_ltl(right, AP):
            mutants.add(binary_factories.get(kind, binary_factories[spot.op_And])(left, m))

mutants

{spot.formula("0"),
 spot.formula("1"),
 spot.formula("p"),
 spot.formula("q"),
 spot.formula("Gp"),
 spot.formula("Gq"),
 spot.formula("!p | Fq"),
 spot.formula("G(!p | Fq)"),
 spot.formula("XG(!p | Fq)"),
 spot.formula("FG(!p | Fq)"),
 spot.formula("X(!p | Fq)"),
 spot.formula("p U G(!p | Fq)"),
 spot.formula("p W G(!p | Fq)"),
 spot.formula("p & G(!p | Fq)"),
 spot.formula("p | G(!p | Fq)"),
 spot.formula("q U G(!p | Fq)"),
 spot.formula("q W G(!p | Fq)"),
 spot.formula("q & G(!p | Fq)"),
 spot.formula("q | G(!p | Fq)"),
 spot.formula("!G(!p | Fq)"),
 spot.formula("!(!p | Fq)"),
 spot.formula("F(!p | Fq)"),
 spot.formula("G!p"),
 spot.formula("GFq"),
 spot.formula("G(!p | q)"),
 spot.formula("G!(!p | Fq)"),
 spot.formula("GF(!p | Fq)"),
 spot.formula("G(!p U Fq)"),
 spot.formula("G(!p R Fq)"),
 spot.formula("G(!p W Fq)"),
 spot.formula("G(!p & Fq)"),
 spot.formula("G(p | Fq)"),
 spot.formula("G(q | Fq)"),
 spot.formula("G(!p | q | Fq)"),
 spot.formula("G(Fq | X!p)"),
 spot.formula("

In [ ]:
"""
LTL Formula Mutation — Definition 9
====================================
Implements mutate(φ, AP) returning the *complete set* of one-step syntactic
mutations of an LTL formula, following Definition 9 rule-by-rule.

Operator encoding
-----------------
  Unary  : 'neg'  (¬)   'next' (○)   'F' (◇)   'G' (□)
  Binary : 'or'   (∨)   'and'  (∧)   'U'        'R'      'W'

Quick start
-----------
    ap  = ('p', 'q')
    phi = UnaryOp('G', Atom('p'))          # □p
    for m in sorted(mutate(phi, ap), key=repr):
        print(m)
"""

from __future__ import annotations
from typing import Sequence, Set

# ---------------------------------------------------------------------------
# Operator pools
# ---------------------------------------------------------------------------
UNARY_OPS  = frozenset({'neg', 'next', 'F', 'G'})      # ¬  ○  ◇  □
BINARY_OPS = frozenset({'or', 'and', 'U', 'R', 'W'})   # ∨  ∧  U  R  W
BINARY_3D  = frozenset({'U', 'W', 'and', 'or'})        # subset for Case 3(d)

_SYM: dict[str, str] = {
    'neg': '¬', 'next': '○', 'F': '◇', 'G': '□',
    'or':  '∨', 'and':  '∧', 'U': 'U', 'R': 'R', 'W': 'W',
}


# ---------------------------------------------------------------------------
# Formula AST
# ---------------------------------------------------------------------------

class Formula:
    """Abstract base for all LTL formula nodes."""
    def mutants(self, ap: Sequence[str]) -> Set['Formula']:
        """Convenience wrapper: return mutate(self, ap)."""
        return mutate(self, ap)


class Const(Formula):
    """Boolean constant — Const(True) = true, Const(False) = false."""
    __slots__ = ('value',)

    def __init__(self, value: bool) -> None:
        self.value = bool(value)

    def __repr__(self) -> str:
        return 'true' if self.value else 'false'

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Const) and self.value == other.value

    def __hash__(self) -> int:
        return hash(('Const', self.value))


class Atom(Formula):
    """Atomic proposition  p ∈ AP."""
    __slots__ = ('name',)

    def __init__(self, name: str) -> None:
        self.name = str(name)

    def __repr__(self) -> str:
        return self.name

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Atom) and self.name == other.name

    def __hash__(self) -> int:
        return hash(('Atom', self.name))


class UnaryOp(Formula):
    """Unary formula  op child,  op ∈ {'neg','next','F','G'}."""
    __slots__ = ('op', 'child')

    def __init__(self, op: str, child: Formula) -> None:
        if op not in UNARY_OPS:
            raise ValueError(f"Unary op must be one of {sorted(UNARY_OPS)}, got {op!r}")
        self.op    = op
        self.child = child

    def __repr__(self) -> str:
        # Parenthesise binary sub-formulas for unambiguous display
        c = f'({self.child})' if isinstance(self.child, BinaryOp) else repr(self.child)
        return f'{_SYM[self.op]}{c}'

    def __eq__(self, other: object) -> bool:
        return (isinstance(other, UnaryOp)
                and self.op == other.op
                and self.child == other.child)

    def __hash__(self) -> int:
        return hash(('UnaryOp', self.op, self.child))


class BinaryOp(Formula):
    """Binary formula  left op right,  op ∈ {'or','and','U','R','W'}."""
    __slots__ = ('op', 'left', 'right')

    def __init__(self, op: str, left: Formula, right: Formula) -> None:
        if op not in BINARY_OPS:
            raise ValueError(f"Binary op must be one of {sorted(BINARY_OPS)}, got {op!r}")
        self.op    = op
        self.left  = left
        self.right = right

    def __repr__(self) -> str:
        return f'({self.left} {_SYM[self.op]} {self.right})'

    def __eq__(self, other: object) -> bool:
        return (isinstance(other, BinaryOp)
                and self.op == other.op
                and self.left == other.left
                and self.right == other.right)

    def __hash__(self) -> int:
        return hash(('BinaryOp', self.op, self.left, self.right))


# ---------------------------------------------------------------------------
# mutate — Definition 9
# ---------------------------------------------------------------------------

def mutate(phi: Formula, ap: Sequence[str]) -> Set[Formula]:
    """
    Return the complete set of mutations of φ following Definition 9 exactly.

    Every applicable rule from the definition is applied; the original
    formula φ is removed from the result (a mutation must be distinct).

    Parameters
    ----------
    phi : Formula
        The LTL formula to mutate.
    ap  : Sequence[str]
        The alphabet of atomic proposition names (AP).

    Returns
    -------
    Set[Formula]
        All φ' produced by Definition 9 with φ itself excluded.
    """
    ms: Set[Formula] = set()

    # ── General Case 5 ── φ' = ○₁ φ,  ○₁ ∈ {¬, ○, ◇, □}
    for op in UNARY_OPS:
        ms.add(UnaryOp(op, phi))


    # ── Base Case 1 ── φ ∈ {true, false}
    if isinstance(phi, Const):
        # true → false  or  false → true
        ms.add(Const(not phi.value))

    # ── Base Case 2 ── φ = p ∈ AP
    elif isinstance(phi, Atom):
        # substitute with any other q ∈ AP, q ≠ p
        for q in ap:
            if q != phi.name:
                ms.add(Atom(q))

    # ── Inductive Case 3 ── φ = ○₁ φ₁
    elif isinstance(phi, UnaryOp):
        phi1, op1 = phi.child, phi.op

        # 3(a) replace unary operator:  ○₁' φ₁,  ○₁' ∈ {¬,○,◇,□} \ {○₁}
        for op in UNARY_OPS:
            if op != op1:
                ms.add(UnaryOp(op, phi1))

        # 3(b) drop the operator:  φ₁
        ms.add(phi1)

        # 3(c) recurse into operand:  ○₁ mutate(φ₁)
        for m1 in mutate(phi1, ap):
            ms.add(UnaryOp(op1, m1))

        # 3(d) embed in binary with a fresh atom on the left:
        #      p ○₂' φ,  p ∈ AP,  ○₂' ∈ {U, W, ∧, ∨}
        for p in ap:
            for op2 in BINARY_3D:
                ms.add(BinaryOp(op2, Atom(p), phi))

    # ── Inductive Case 4 ── φ = φ₁ ○₂ φ₂
    elif isinstance(phi, BinaryOp):
        phi1, phi2, op2 = phi.left, phi.right, phi.op

        # 4(a) replace binary operator:  φ₁ ○₂' φ₂,  ○₂' ∈ {∨,∧,U,R,W} \ {○₂}
        for op in BINARY_OPS:
            if op != op2:
                ms.add(BinaryOp(op, phi1, phi2))

        # 4(b) project to one sub-formula:  φ₁  or  φ₂
        ms.add(phi1)
        ms.add(phi2)

        # 4(c) recurse into left operand:  mutate(φ₁) ○₂ φ₂
        for m1 in mutate(phi1, ap):
            ms.add(BinaryOp(op2, m1, phi2))

        # 4(d) recurse into right operand:  φ₁ ○₂ mutate(φ₂)
        for m2 in mutate(phi2, ap):
            ms.add(BinaryOp(op2, phi1, m2))

    # A mutation must be syntactically distinct from the original formula
    ms.discard(phi)
    return ms


# ---------------------------------------------------------------------------
# Demo
# ---------------------------------------------------------------------------
if __name__ == '__main__':
    AP = ('p', 'q')

    examples: list[tuple[str, Formula]] = [
        ('true',       Const(True)),
        ('false',      Const(False)),
        ('p',          Atom('p')),
        ('¬p',         UnaryOp('neg',  Atom('p'))),
        ('□p',         UnaryOp('G',    Atom('p'))),
        ('◇p',         UnaryOp('F',    Atom('p'))),
        ('p U q',      BinaryOp('U',   Atom('p'), Atom('q'))),
        ('p ∨ q',      BinaryOp('or',  Atom('p'), Atom('q'))),
        ('□(p U q)',   UnaryOp('G',    BinaryOp('U', Atom('p'), Atom('q')))),
    ]

    for label, phi in examples:
        ms = mutate(phi, AP)
        print(f'\nφ = {label}  →  {len(ms)} mutant(s):')
        for m in sorted(ms, key=repr):
            print(f'   {m}')


φ = true  →  7 mutant(s):
   false
   p
   q
   ¬true
   □true
   ◇true
   ○true

φ = false  →  7 mutant(s):
   p
   q
   true
   ¬false
   □false
   ◇false
   ○false

φ = p  →  7 mutant(s):
   false
   q
   true
   ¬p
   □p
   ◇p
   ○p

φ = ¬p  →  25 mutant(s):
   (p U ¬p)
   (p W ¬p)
   (p ∧ ¬p)
   (p ∨ ¬p)
   (q U ¬p)
   (q W ¬p)
   (q ∧ ¬p)
   (q ∨ ¬p)
   false
   p
   q
   true
   ¬false
   ¬q
   ¬true
   ¬¬p
   ¬□p
   ¬◇p
   ¬○p
   □p
   □¬p
   ◇p
   ◇¬p
   ○p
   ○¬p

φ = □p  →  25 mutant(s):
   (p U □p)
   (p W □p)
   (p ∧ □p)
   (p ∨ □p)
   (q U □p)
   (q W □p)
   (q ∧ □p)
   (q ∨ □p)
   false
   p
   q
   true
   ¬p
   ¬□p
   □false
   □q
   □true
   □¬p
   □□p
   □◇p
   □○p
   ◇p
   ◇□p
   ○p
   ○□p

φ = ◇p  →  25 mutant(s):
   (p U ◇p)
   (p W ◇p)
   (p ∧ ◇p)
   (p ∨ ◇p)
   (q U ◇p)
   (q W ◇p)
   (q ∧ ◇p)
   (q ∨ ◇p)
   false
   p
   q
   true
   ¬p
   ¬◇p
   □p
   □◇p
   ◇false
   ◇q
   ◇true
   ◇¬p
   ◇□p
   ◇◇p
   ◇○p
   ○p
   ○◇p

φ = p U q  →  26 mutant(s):
   (false 

In [17]:
import spot

class LTLMutator:
    def __init__(self, ap_list):
        """
        Initialize the mutator with a set of atomic propositions.
        
        :param ap_list: A list of strings representing atomic propositions (e.g., ['p', 'q'])
        """
        self.ap_set = set(ap_list)
        self.aps = [spot.formula.ap(p) for p in self.ap_set]
        
        # Map Spot operator kinds to their constructor functions (Rule 3)
        self.unary_ops = {
            spot.op_Not: spot.formula.Not,
            spot.op_X: spot.formula.X,
            spot.op_F: spot.formula.F,
            spot.op_G: spot.formula.G
        }
        
        # Map Spot operator kinds to their constructor functions (Rule 4)
        self.binary_ops = {
            spot.op_U: spot.formula.U,
            spot.op_R: spot.formula.R,
            spot.op_W: spot.formula.W,
            spot.op_And: lambda a, b: spot.formula.And([a, b]),
            spot.op_Or: lambda a, b: spot.formula.Or([a, b])
        }
        
        # Binary operators specifically allowed for appending in Rule 3(d)
        self.rule3d_ops = {
            'U': spot.formula.U,
            'W': spot.formula.W,
            '&': lambda a, b: spot.formula.And([a, b]),
            '|': lambda a, b: spot.formula.Or([a, b])
        }

    def mutate(self, phi):
        """
        Returns a list of all unique 1-point mutations for the given LTL formula.
        
        :param phi: A string or spot.formula object
        :return: A list of spot.formula objects representing the mutations
        """
        if isinstance(phi, str):
            phi = spot.formula(phi)
            
        # Definition 9 relies on core LTL operators. If the user provides implies/equiv, 
        # unabbreviate them first so they fit into the binary_ops logic.
        phi = phi.unabbreviate("ie")
            
        mutations = {}
        for mut_f in self._mutate_recursive(phi):
            s = mut_f.to_str()
            # Deduplicate by string representation
            if s not in mutations:
                mutations[s] = mut_f
                
        # A valid mutation should be a modification, not the original formula itself.
        orig_s = phi.to_str()
        if orig_s in mutations:
            del mutations[orig_s]
            
        return list(mutations.values())

    def _mutate_recursive(self, f):
        """
        Recursive generator for producing mutations based on Definition 9.
        """
        # --- GENERAL CASES ---
        # 5. phi' = o_1 phi
        for op_func in self.unary_ops.values():
            yield op_func(f)
            
        # 6. phi' = x, where x in {true, false} U AP
        yield spot.formula.tt()
        yield spot.formula.ff()
        for p in self.aps:
            yield p
            
        kind = f.kind()
        
        # --- BASE CASES ---
        # 1. If phi = true

In [18]:
# 1. Define the pool of atomic propositions used in your system
ap_pool = ['p', 'q']

# 2. Instantiate the Mutator
mutator = LTLMutator(ap_pool)

# 3. Supply the base LTL formula you wish to mutate
original_formula = "G(p U q)"
mutations = mutator.mutate(original_formula)

print(f"Original Formula: {original_formula}")
print(f"Total Unique Mutations Found: {len(mutations)}")
print("-" * 30)

# 4. Display the results
for i, m in enumerate(mutations):
    print(f"{i + 1:02d}: {m.to_str()}")

Original Formula: G(p U q)
Total Unique Mutations Found: 7
------------------------------
01: !G(p U q)
02: XG(p U q)
03: FG(p U q)
04: 1
05: 0
06: p
07: q


In [11]:
import spot

class LTLMutator:
    def __init__(self, ap_list):
        self.ap_set = set(ap_list)
        self.aps = [spot.formula.ap(p) for p in self.ap_set]
        
        self.unary_ops = {
            spot.op_Not: spot.formula.Not,
            spot.op_X: spot.formula.X,
            spot.op_F: spot.formula.F,
            spot.op_G: spot.formula.G
        }
        
        self.binary_ops = {
            spot.op_U: spot.formula.U,
            spot.op_R: spot.formula.R,
            spot.op_W: spot.formula.W,
            spot.op_And: lambda a, b: spot.formula.And([a, b]),
            spot.op_Or: lambda a, b: spot.formula.Or([a, b])
        }
        
        self.rule3d_ops = {
            'U': spot.formula.U,
            'W': spot.formula.W,
            '&': lambda a, b: spot.formula.And([a, b]),
            '|': lambda a, b: spot.formula.Or([a, b])
        }

    def mutate(self, phi):
        if isinstance(phi, str):
            phi = spot.formula(phi)
            
        phi = phi.unabbreviate("ie")
        mutations = {}
        
        for mut_f in self._mutate_recursive(phi):
            s = mut_f.to_str()
            if s not in mutations:
                mutations[s] = mut_f
                
        orig_s = phi.to_str()
        if orig_s in mutations:
            del mutations[orig_s]
            
        return list(mutations.values())

    def _mutate_recursive(self, f):
        # --- GENERAL CASES ---
        for op_func in self.unary_ops.values():
            yield op_func(f)
            
        yield spot.formula.tt()
        yield spot.formula.ff()
        for p in self.aps:
            yield p
            
        kind = f.kind()
        
        # --- BASE CASES ---
        if f.is_tt():
            yield spot.formula.ff()
            
        if f in self.ap_set:
            for p in self.aps:
                if p != f:
                    yield p
                    
        # --- INDUCTIVE CASES ---
        if kind in self.unary_ops:
            child = f[0]
            
            # 3(a) Change unary operator
            for other_kind, op_func in self.unary_ops.items():
                if other_kind != kind:
                    yield op_func(child)
                    
            # 3(b) Drop operator
            yield child
            
            # 3(c) Mutate child -> CRITICAL FIX: Added 'yield from'
            for mutated_child in self._mutate_recursive(child):
                yield self.unary_ops[kind](mutated_child)
                
            # 3(d) Append binary operator
            for p in self.aps:
                for op_func in self.rule3d_ops.values():
                    yield op_func(p, f)
                    
        elif kind in self.binary_ops:
            children = list(f)
            
            if len(children) >= 2:
                phi_1 = children[0]
                if len(children) > 2:
                    if kind == spot.op_And:
                        phi_2 = spot.formula.And(children[1:])
                    elif kind == spot.op_Or:
                        phi_2 = spot.formula.Or(children[1:])
                    else:
                        phi_2 = children[1]
                else:
                    phi_2 = children[1]
                    
                # 4(a) Change binary operator
                for other_kind, op_func in self.binary_ops.items():
                    if other_kind != kind:
                        yield op_func(phi_1, phi_2)
                        
                # 4(b) Keep one child
                yield phi_1
                yield phi_2
                
                # 4(c) Mutate left child -> CRITICAL FIX: Added 'yield from'
                for mutated_left in self._mutate_recursive(phi_1):
                    yield self.binary_ops[kind](mutated_left, phi_2)
                    
                # 4(d) Mutate right child -> CRITICAL FIX: Added 'yield from'
                for mutated_right in self._mutate_recursive(phi_2):
                    yield self.binary_ops[kind](phi_1, mutated_right)
                    
    def _mutate_recursive(self, f, is_top_level=True):
        # --- GENERAL CASES (Strictly restricted based on your structural goal) ---
        # Rule 5: Wrap the current sub-formula in a unary operator
        for op_func in self.unary_ops.values():
            yield op_func(f)
            
        kind = f.kind()
        
        # --- BASE CASES / LEAF MUTATIONS ---
        # Rule 1 & Rule 6 (Moved here so they only mutate leaves, not the whole tree)
        if f.is_tt():
            yield spot.formula.ff()
            
        if f.is_ff():
            yield spot.formula.tt()
            
        if f.is_tt():
            yield spot.formula.ff()

        if f in self.ap_set:
            # Rule 2: Swap APs
            for p in self.aps:
                if p != f:
                    yield p
            # Rule 6 for APs: allow mapping an AP to other APs (handled above)
        
        # If we are at the top level of a complex formula, we block Rule 6 
        # from replacing the entire tree with a single 'p' or 'true'.
        if not is_top_level or f in self.ap_set or f.is_tt() or f.is_ff():
            for p in self.aps:
                yield p

        # --- INDUCTIVE CASES ---
        if kind in self.unary_ops:
            child = f[0]
            
            # 3(a) Change unary operator
            for other_kind, op_func in self.unary_ops.items():
                if other_kind != kind:
                    yield op_func(child)
                    
            # 3(b) Drop operator
            yield child
            
            # 3(c) Mutate child (Passes is_top_level=False)
            for mutated_child in self._mutate_recursive(child, is_top_level=False):
                yield self.unary_ops[kind](mutated_child)
                
            # 3(d) Append binary operator
            for p in self.aps:
                for op_func in self.rule3d_ops.values():
                    yield op_func(p, f)
                    
        elif kind in self.binary_ops:
            children = list(f)
            
            if len(children) >= 2:
                phi_1 = children[0]
                if len(children) > 2:
                    if kind == spot.op_And:
                        phi_2 = spot.formula.And(children[1:])
                    elif kind == spot.op_Or:
                        phi_2 = spot.formula.Or(children[1:])
                    else:
                        phi_2 = children[1]
                else:
                    phi_2 = children[1]
                    
                # 4(a) Change binary operator
                for other_kind, op_func in self.binary_ops.items():
                    if other_kind != kind:
                        yield op_func(phi_1, phi_2)
                        
                # 4(b) Keep one child
                yield phi_1
                yield phi_2
                
                # 4(c) Mutate left child (Passes is_top_level=False)
                for mutated_left in self._mutate_recursive(phi_1, is_top_level=False):
                    yield self.binary_ops[kind](mutated_left, phi_2)
                    
                # 4(d) Mutate right child (Passes is_top_level=False)
                for mutated_right in self._mutate_recursive(phi_2, is_top_level=False):
                    yield self.binary_ops[kind](phi_1, mutated_right)

In [12]:
# 1. Define the pool of atomic propositions used in your system
ap_pool = ['p', 'q']

# 2. Instantiate the Mutator
mutator = LTLMutator(ap_pool)

# 3. Supply the base LTL formula you wish to mutate
original_formula = "G(p U q)"
mutations = mutator.mutate(original_formula)

print(f"Original Formula: {original_formula}")
print(f"Total Unique Mutations Found: {len(mutations)}")
print("-" * 30)

# 4. Display the results
for i, m in enumerate(mutations):
    print(f"{i + 1:02d}: {m.to_str()}")

Original Formula: G(p U q)
Total Unique Mutations Found: 32
------------------------------
01: !G(p U q)
02: XG(p U q)
03: FG(p U q)
04: !(p U q)
05: X(p U q)
06: F(p U q)
07: p U q
08: G!(p U q)
09: GX(p U q)
10: GF(p U q)
11: Gp
12: Gq
13: G(p R q)
14: G(p W q)
15: G(p & q)
16: G(p | q)
17: G(!p U q)
18: G(Xp U q)
19: G(Fp U q)
20: G(Gp U q)
21: G(p U !q)
22: G(p U Xq)
23: G(p U Fq)
24: G(p U Gq)
25: p U G(p U q)
26: p W G(p U q)
27: p & G(p U q)
28: p | G(p U q)
29: q U G(p U q)
30: q W G(p U q)
31: q & G(p U q)
32: q | G(p U q)


In [13]:
from collections import deque
import spot

# Reuse the structural LTLMutator class we refined in the previous step
class LTLMutator:
    def __init__(self, ap_list):
        self.ap_set = set(ap_list)
        self.aps = [spot.formula.ap(p) for p in self.ap_set]
        self.unary_ops = {
            spot.op_Not: spot.formula.Not, spot.op_X: spot.formula.X,
            spot.op_F: spot.formula.F, spot.op_G: spot.formula.G
        }
        self.binary_ops = {
            spot.op_U: spot.formula.U, spot.op_R: spot.formula.R, spot.op_W: spot.formula.W,
            spot.op_And: lambda a, b: spot.formula.And([a, b]),
            spot.op_Or: lambda a, b: spot.formula.Or([a, b])
        }
        self.rule3d_ops = {
            'U': spot.formula.U, 'W': spot.formula.W,
            '&': lambda a, b: spot.formula.And([a, b]), '|': lambda a, b: spot.formula.Or([a, b])
        }

    def mutate(self, phi):
        if isinstance(phi, str):
            phi = spot.formula(phi)
        phi = phi.unabbreviate("ie")
        mutations = {}
        for mut_f in self._mutate_recursive(phi, is_top_level=True):
            s = mut_f.to_str()
            if s not in mutations:
                mutations[s] = mut_f
        orig_s = phi.to_str()
        if orig_s in mutations:
            del mutations[orig_s]
        return list(mutations.values())

    def _mutate_recursive(self, f, is_top_level=True):
        for op_func in self.unary_ops.values():
            yield op_func(f)
            
        if f.is_tt(): yield spot.formula.ff()
        if f.is_ff() or f in self.ap_set: yield spot.formula.tt()
        if f.is_tt() or f in self.ap_set: yield spot.formula.ff()

        if f in self.ap_set:
            for p in self.aps:
                if p != f: yield p
        
        if not is_top_level or f in self.ap_set or f.is_tt() or f.is_ff():
            for p in self.aps: yield p

        kind = f.kind()
        if kind in self.unary_ops:
            child = f[0]
            for other_kind, op_func in self.unary_ops.items():
                if other_kind != kind: yield op_func(child)
            yield child
            for mutated_child in self._mutate_recursive(child, is_top_level=False):
                yield self.unary_ops[kind](mutated_child)
            for p in self.aps:
                for op_func in self.rule3d_ops.values(): yield op_func(p, f)
                    
        elif kind in self.binary_ops:
            children = list(f)
            if len(children) >= 2:
                phi_1 = children[0]
                phi_2 = spot.formula.And(children[1:]) if kind == spot.op_And else (spot.formula.Or(children[1:]) if kind == spot.op_Or else children[1])
                    
                for other_kind, op_func in self.binary_ops.items():
                    if other_kind != kind: yield op_func(phi_1, phi_2)
                yield phi_1
                yield phi_2
                for mutated_left in self._mutate_recursive(phi_1, is_top_level=False):
                    yield self.binary_ops[kind](mutated_left, phi_2)
                for mutated_right in self._mutate_recursive(phi_2, is_top_level=False):
                    yield self.binary_ops[kind](phi_1, mutated_right)


class LTLMutationDistance:
    def __init__(self, ap_list):
        """
        Initialize the distance calculator with the allowed atomic propositions.
        """
        self.mutator = LTLMutator(ap_list)

    def calculate_distance(self, source, target):
        """
        Calculates the minimum mutation distance from source formula to target formula.
        
        :param source: Starting LTL formula string (e.g., "G(p U q)")
        :param target: Destination LTL formula string (e.g., "p U !q")
        :return: (int, list) The distance, and the step-by-step path taken. Returns (inf, []) if unreachable.
        """
        src_formula = spot.formula(source).unabbreviate("ie")
        tgt_formula = spot.formula(target).unabbreviate("ie")
        
        src_str = src_formula.to_str()
        tgt_str = tgt_formula.to_str()
        
        if src_str == tgt_str:
            return 0, [src_str]
            
        # BFS Queue holds tuples of (current_formula, path_taken_as_list)
        queue = deque([(src_formula, [src_str])])
        
        # Visited set tracks string representations to prevent infinite loops
        visited = {src_str}
        
        while queue:
            current_formula, current_path = queue.popleft()
            
            # Generate all valid 1-point mutations from the current formula
            neighbors = self.mutator.mutate(current_formula)
            
            for neighbor in neighbors:
                neighbor_str = neighbor.to_str()
                
                if neighbor_str == tgt_str:
                    return len(current_path), current_path + [neighbor_str]
                    
                if neighbor_str not in visited:
                    visited.add(neighbor_str)
                    queue.append((neighbor, current_path + [neighbor_str]))
                    
        return float('inf'), [] # If no structural mutation path connects them

In [14]:
# Define the AP pool
ap_pool = ['p', 'q']
dist_calculator = LTLMutationDistance(ap_pool)

# Define source and destination formulas
start = "GF(p U q)"
end = "p U !(q->p)"

distance, path = dist_calculator.calculate_distance(start, end)

print(f"Source: {start}")
print(f"Target: {end}")
print(f"Mutation Distance: {distance}")
print("Optimal Mutation Path:")
print(" -> ".join(path))

Source: GF(p U q)
Target: p U !(q->p)
Mutation Distance: 5
Optimal Mutation Path:
GF(p U q) -> !GF(p U q) -> !q -> p U !q -> p U (p | !q) -> p U !(p | !q)


In [16]:
from collections import deque
import spot

class LTLTree:
    """A pure, un-optimized Abstract Syntax Tree (AST) for LTL formulas."""
    def __init__(self, value, children=None):
        self.value = value  # e.g., 'G', 'F', '!', 'U', '&', 'p', 'q'
        self.children = children if children is not None else []

    @classmethod
    def from_spot(cls, f):
        """Recursively parses a Spot formula into a pure text AST."""
        kind = f.kind()
        
        if f.is_tt(): return cls("true")
        if f.is_ff(): return cls("false")
        
        # Check if Atomic Proposition
        if kind == spot.op_ap: 
            return cls(f.to_str())
        
        # Handle Unary Operators
        if kind in [spot.op_Not, spot.op_X, spot.op_F, spot.op_G]:
            op_str = {spot.op_Not: '!', spot.op_X: 'X', spot.op_F: 'F', spot.op_G: 'G'}[kind]
            return cls(op_str, [cls.from_spot(f[0])])
            
        # Handle Binary/N-ary Operators
        if kind in [spot.op_U, spot.op_R, spot.op_W, spot.op_And, spot.op_Or]:
            op_str = {spot.op_U: 'U', spot.op_R: 'R', spot.op_W: 'W', spot.op_And: '&', spot.op_Or: '|'}[kind]
            children = list(f)
            
            # FIX HERE: Manually fold flat multi-operand structures (e.g., [a, b, c]) 
            # into right-nested binary structures (e.g., (a & (b & c)))
            if len(children) > 2 and kind in [spot.op_And, spot.op_Or]:
                # Start from the last element and build upwards
                right_child = cls.from_spot(children[-1])
                for child in reversed(children[1:-1]):
                    right_child = cls(op_str, [cls.from_spot(child), right_child])
                return cls(op_str, [cls.from_spot(children[0]), right_child])
                
            return cls(op_str, [cls.from_spot(children[0]), cls.from_spot(children[1])])
            
        raise ValueError(f"Unsupported formula component: {f.to_str()}")

    def to_str(self):
        """Converts the tree back to a cleanly parenthesized string format."""
        if not self.children:
            return self.value
        if len(self.children) == 1:
            # Unary operators: e.g., !GF(p U q) -> !(G(F(p U q)))
            child_str = self.children[0].to_str()
            if len(self.value) > 1 or self.value.isalpha() or child_str.startswith('('):
                return f"{self.value}{child_str}"
            return f"{self.value}({child_str})"
        else:
            # Binary operators
            return f"({self.children[0].to_str()} {self.value} {self.children[1].to_str()})"

    def copy(self):
        return LTLTree(self.value, [c.copy() for c in self.children])


class PureLTLMutator:
    def __init__(self, ap_list):
        self.aps = ap_list
        self.unary_ops = ['!', 'X', 'F', 'G']
        self.binary_ops = ['U', 'R', 'W', '&', '|']

    def mutate(self, tree):
        mutations = set()
        for mut_tree in self._mutate_recursive(tree, is_top_level=True):
            mutations.add(mut_tree.to_str())
        orig_str = tree.to_str()
        if orig_str in mutations:
            mutations.remove(orig_str)
        return list(mutations)

    def _mutate_recursive(self, node, is_top_level=True):
        # --- GENERAL CASES ---
        # Rule 5: Wrap in unary operator
        for op in self.unary_ops:
            yield LTLTree(op, [node.copy()])
            
        # Rule 1 & 6: Constants/Leaves (Only at leaf level or structurally allowed)
        if node.value == "true": yield LTLTree("false")
        if node.value in ["false"] + self.aps: yield LTLTree("true")
        if node.value in ["true"] + self.aps: yield LTLTree("false")
        
        if node.value in self.aps:
            for p in self.aps:
                if p != node.value: yield LTLTree(p)
                
        if not is_top_level or node.value in (self.aps + ["true", "false"]):
            for p in self.aps:
                yield LTLTree(p)

        # --- INDUCTIVE CASES ---
        if node.value in self.unary_ops:
            child = node.children[0]
            
            # 3(a) Change unary operator
            for op in self.unary_ops:
                if op != node.value: yield LTLTree(op, [child.copy()])
            # 3(b) Drop operator
            yield child.copy()
            # 3(c) Mutate child
            for mut_child in self._mutate_recursive(child, is_top_level=False):
                yield LTLTree(node.value, [mut_child])
            # 3(d) Append binary operator
            for p in self.aps:
                for op in ['U', 'W', '&', '|']:
                    yield LTLTree(op, [LTLTree(p), node.copy()])

        elif node.value in self.binary_ops:
            left, right = node.children[0], node.children[1]
            
            # 4(a) Change binary operator
            for op in self.binary_ops:
                if op != node.value: yield LTLTree(op, [left.copy(), right.copy()])
            # 4(b) Keep one child
            yield left.copy()
            yield right.copy()
            # 4(c) Mutate left child
            for mut_left in self._mutate_recursive(left, is_top_level=False):
                yield LTLTree(node.value, [mut_left, right.copy()])
            # 4(d) Mutate right child
            for mut_right in self._mutate_recursive(right, is_top_level=False):
                yield LTLTree(node.value, [left.copy(), mut_right])


class PureLTLDistanceCalculator:
    def __init__(self, ap_list):
        self.mutator = PureLTLMutator(ap_list)

    def distance(self, start_formula, end_formula):
        # Normalize target string structure via the exact same syntax tree format
        src_tree = LTLTree.from_spot(spot.formula(start_formula))
        tgt_tree = LTLTree.from_spot(spot.formula(end_formula))
        
        src_str = src_tree.to_str()
        tgt_str = tgt_tree.to_str()
        
        if src_str == tgt_str:
            return 0, [src_str]
            
        queue = deque([(src_tree, [src_str])])
        visited = {src_str}
        
        while queue:
            curr_tree, path = queue.popleft()
            neighbors = self.mutator.mutate(curr_tree)
            
            for n_str in neighbors:
                if n_str == tgt_str:
                    return len(path), path + [n_str]
                if n_str not in visited:
                    visited.add(n_str)
                    queue.append((LTLTree.from_spot(spot.formula(n_str)), path + [n_str]))
                    
        return float('inf'), []
    

calc = PureLTLDistanceCalculator(['p', 'q'])
# Note: "q->p" maps syntactically to "!q | p" under standard unabbreviation tree formats
dist, path = calc.distance("GF(p U q)", "p U !( !q | p )")

print(f"Mutation Distance: {dist}")
print("Strict Structural Path:")
for step in path:
    print(f" -> {step}")

Mutation Distance: 5
Strict Structural Path:
 -> GF(p U q)
 -> (q | GF(p U q))
 -> (!(q) | GF(p U q))
 -> (!(q) | p)
 -> !(p | !(q))
 -> (p U !(p | !(q)))
